# 2026 COMP90042 Project Group 141 — Per-Evidence BERT + Score-Weighted Aggregation

*Extends hybrid 2-stage retrieval with a designed classification stack (Tier 1).*

**Retrieval:** TF-IDF + dense Stage 1 → cross-encoder rerank (unchanged from hybrid notebook).

**Classification (this notebook):**
1. **Per-evidence BERT** — one (claim, passage) pair per forward pass
2. **Score-weighted logit aggregation** — combine passage votes using reranker scores
3. **Mixed training** — 50% pair examples from gold evidence + 50% from retrieved train evidence
4. **Retrieved-dev validation** — early stopping on dev pairs from retrieved (not gold) passages
5. **Class-weighted loss** — handles label imbalance

**Baseline comparison:** `2stageHybrid_RetrievedTrainBERT.ipynb` (concatenated multi-evidence BERT).


In [ ]:
# Readme

Run **top-to-bottom** on Colab (GPU). Set `data_path` after mounting Drive.

## Flow (what changed vs hybrid notebook)

```text
RETRIEVAL (same as hybrid)
  claim → Stage1 (TF-IDF + dense) → cross-encoder rerank → top-k passages + scores

TRAINING (NEW)
  each (claim, single passage) = one BERT row
  mix ~50% rows from GOLD passages + ~50% from RETRIEVED passages
  label always = gold claim_label from JSON

INFERENCE (NEW)
  for each passage: BERT → 4 logits
  combine logits with softmax(reranker scores) → one claim label
```

| Flag | Default | Meaning |
|------|---------|---------|
| `MIXED_TRAIN_RATIO` | `0.5` | Fraction of gold-evidence pair rows in mixed training |
| `USE_RETRIEVED_DEV_EVAL` | `True` | Validate on dev **retrieved** pair examples |
| `USE_CLASS_WEIGHTS` | `True` | Weighted cross-entropy in Trainer |

**Code comments:** search for `# NEW` and `# =====` in cells 10, 29–30, 33–36.

**Report ablations:** concat BERT (hybrid) vs per-evidence + aggregation (this notebook).

**Final test:** `USE_TRAIN_PLUS_DEV_FOR_FINAL = True` only after locking dev metrics; then run final + test cells.


# Readme
*Run top-to-bottom on Google Colab (GPU). Mount Drive and set `data_path` in cell 7.*

**Tuning:** Change `STAGE1_TOP_N_DEFAULT` / `RERANK_TOP_K_DEFAULT` in cell 15, then re-run from Stage 1 cache (cell 20) if N changes.

**Final test only:** After dev metrics are fixed, set `USE_TRAIN_PLUS_DEV_FOR_FINAL = True` in cell 9, then run cells 40–41. Keep it `False` while tuning on dev (README §1 Data Usage).

### README compliance (§2 Important Notes)
| Rule | How this notebook complies |
|------|---------------------------|
| Transformer required (§2.1) | `google-bert/bert-base-uncased` fine-tuned via Hugging Face `Trainer` |
| Open-source / Colab only (§2.2, §2.7) | HF models only: BERT, `multi-qa-MiniLM-L6-cos-v1`, `ms-marco-MiniLM-L6-v2`; no closed APIs |
| No hand-crafted label rules (§2.3) | Labels from learned classifiers only; `if` flags are pipeline config, not prediction rules |
| Allowed libraries (§2.4) | PyTorch, Hugging Face, NumPy, scikit-learn (TF-IDF), sentence-transformers (pretrained encoders) |
| Provided data only (§2.9) | `train-claims.json`, `dev-claims.json`, `evidence.json` only; labels/evidence from these files |
| Template sections (§2.8) | §1 / §2.1 / §2.2 / §3 titles unchanged; extra cells added below sections |
| Train+dev for test (§1 Data Usage) | Optional cell 40 merges train+dev **only** when `USE_TRAIN_PLUS_DEV_FOR_FINAL=True` |
| No test inspection / no manual edits (§2.6, leaderboard) | Test has no labels in notebook; `test-output.json` produced by code only |
| Hybrid design (§2.2 encouragement) | TF-IDF + dense retrieval → cross-encoder rerank → Transformer classifier |


In [ ]:
# Install the requirements in Google Colab
!pip install -q -U transformers accelerate datasets evaluate sentence-transformers scikit-learn


In [ ]:
# Authenticate to Hugging Face
from huggingface_hub import login

login()

In [ ]:
# setting device on GPU if available, else CPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


In [ ]:
# load data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# base data path
data_path = Path("/content/drive/MyDrive/JSON")

# all file paths
data_path_train_claims = data_path / "train-claims.json"
data_path_evidence = data_path / "evidence.json"
data_path_dev_claims = data_path / "dev-claims.json"
data_path_dev_baseline_claims = data_path / "dev-claims-baseline.json"
data_path_test_claims = data_path / "test-claims-unlabelled.json"


In [ ]:
import json

with open(data_path_train_claims) as file:
    train_claims = json.load(file)
# train_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_evidence) as file:
    evidence = json.load(file)
# evidence is a dictionary with evidence_id as key and the actual text
# as value.

with open(data_path_dev_claims) as file:
    dev_claims = json.load(file)
# dev_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_dev_baseline_claims) as file:
    dev_baseline_claims = json.load(file)
# dev_baseline_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_test_claims) as file:
    test_claims = json.load(file)
# test_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores just claim_text.

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def process_data(data, retrieve_train=False, classification_train=False):
    """Process claims into task-specific rows.

    - retrieve_train=True:
      output: list of {claim_id, claim_text}
    - classification_train=True:
      output: list of {claim_id, claim_text, label, evidences, gold_input}
    Exactly one mode should be True.
    """
    processed_data = []
    for claim_id, value in data.items():
        claim_text = clean_text(value["claim_text"])

        if retrieve_train:
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
            })
        elif classification_train:
            evidence_ids = list(value.get("evidences", []))
            evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]
            gold_input = (
                f"Claim: {claim_text} \n\nEvidence:\n"
                + "\n".join([f"- {e}" for e in evidence_texts])
            )
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidences": evidence_ids,
                "label": value.get("claim_label"),
                "gold_input": gold_input,
            })

    return processed_data

def add_gold_input_for_records(records):
    """For list records with evidences, append/build `gold_input` in place."""
    for record in records:
        claim_text = clean_text(record["claim_text"])
        evidence_ids = list(record.get("evidences", []))
        evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]

        gold_input = (
            f"Claim: {claim_text} \n\nEvidence:\n"
            + "\n".join([f"- {e}" for e in evidence_texts])
        )

        record["gold_input"] = gold_input

    return records


# Retrieval training/inference rows.
processed_train_retrieve_data = process_data(train_claims, retrieve_train=True)
processed_dev_retrieve_data = process_data(dev_claims, retrieve_train=True)

# Gold-evidence rows (annotator passages) — used for MIXED training and diagnostics.
processed_train_data = process_data(train_claims, classification_train=True)
processed_dev_data = process_data(dev_claims, classification_train=True)

processed_test_data = process_data(test_claims, retrieve_train=True)

# =============================================================================
# NEW (Tier 1 classification) — toggles explained
# =============================================================================
# MIXED_TRAIN_RATIO:
#   Of all BERT *pair* training rows, what fraction come from GOLD passages vs RETRIEVED?
#   0.5 => ~half rows see perfect evidence, half see what the retriever actually returns.
#
# USE_RETRIEVED_DEV_EVAL:
#   If True, early-stopping uses dev *retrieved* pair rows (matches test-time noise).
#   If False, uses dev *gold* pair rows (easier validation, mismatched with inference).
#
# USE_CLASS_WEIGHTS:
#   Rare labels (e.g. DISPUTED) get higher loss weight so BERT does not ignore them.
#
# USE_TRAIN_PLUS_DEV_FOR_FINAL:
#   Only set True for the final optional retrain + test cells (README allows train+dev for test).
MIXED_TRAIN_RATIO = 0.5
USE_RETRIEVED_DEV_EVAL = True
USE_CLASS_WEIGHTS = True
USE_TRAIN_PLUS_DEV_FOR_FINAL = False


In [ ]:
all_evidence_ids = list(evidence.keys())
all_evidence_texts = [clean_text(evidence[eid]) for eid in all_evidence_ids]
# all_evidence_ids and all_evidence_texts are coresponding lists of all the evidence passages.

In [ ]:
print(f"all_evidence_ids.length = {len(all_evidence_ids)}")
print(f"len(processed_train_data) = {len(processed_train_data)}")
print(f"len(processed_dev_data) = {len(processed_dev_data)}")
print(f"len(processed_test_data) = {len(processed_test_data)}")
print(processed_train_data[0])
print(processed_train_data[0].get("gold_input"))

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


## 2.1 Evidence Retrieval
Given claim_text, output evidence_ids

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import copy
import random
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
from tqdm.auto import tqdm

from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers import SentenceTransformer, util

In [ ]:
STAGE1_TOP_N_DEFAULT = 500
RERANK_TOP_K_DEFAULT = 5  # tuned: slightly higher recall for retrieval F
STAGE1_CLAIM_BATCH_SIZE = 8


### 2.1.1 Stage 1: Find top-k evidence

In [ ]:
st_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

evidence_st = st_model.encode(all_evidence_texts,
                                 batch_size=128,
                                 convert_to_tensor=True,
                                 show_progress_bar=True,
                                 normalize_embeddings=True)

# multi-qa-mpnet-base-dot-v1

# multi-qa-MiniLM-L6-cos-v1

In [ ]:
# Build TF-IDF index over all evidence passages
# vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

evidence_tfidf = vectorizer.fit_transform(all_evidence_texts)

evidence_id_to_text = {
    eid: text for eid, text in zip(all_evidence_ids, all_evidence_texts)
}

In [ ]:
# Input: claim_text
# Return: top-k evidence_ids
def retrieve_stage1_candidates(claim_text_list,top_n=STAGE1_TOP_N_DEFAULT, return_scores=False):
    """Top-N evidence ids per claim. claim_text_list is a list of claim strings.

    TF-IDF is L2-normalized by default, so sparse dot product equals cosine similarity.
    Returns a list with one entry per input claim (each entry is a list of evidence ids,
    or (ids, scores) if return_scores is True).
    """
    if not claim_text_list:
        return []

    claim_clean_list = [clean_text(t) for t in claim_text_list]
    tfidf_k, st_k = top_n, top_n

    # TF-IDF retrival
    claim_tfidf = vectorizer.transform(claim_clean_list)
    tfidf_scores = cosine_similarity(claim_tfidf, evidence_tfidf)


    tfidf_k = min(tfidf_k, tfidf_scores.shape[1])
    tfidf_idx = np.argpartition(-tfidf_scores, tfidf_k - 1, axis=1)[:, :tfidf_k]

    # SentenceTransformer retrival
    claim_emb = st_model.encode(claim_clean_list,
                                convert_to_tensor=True,
                                normalize_embeddings=True,)

    st_scores = util.cos_sim(claim_emb, evidence_st).cpu().numpy()

    st_k = min(st_k, st_scores.shape[1])
    st_idx = np.argpartition(-st_scores, st_k - 1, axis=1)[:, :st_k]

    # Merge SentenceTransformer and TF-IDF
    batch_size = len(claim_text_list)
    results = []
    for b in range(batch_size):
        tfidf_candidates = tfidf_idx[b].tolist()
        dense_candidates = st_idx[b].tolist()

        # union
        merged = list(set(tfidf_candidates) | set(dense_candidates))

        # optional scoring (simple fusion)
        score_map = {}

        for idx in merged:
            score_map[idx] = (
                tfidf_scores[b, idx] + st_scores[b, idx]
            )

        # sort by fused score
        ranked = sorted(merged, key=lambda x: score_map[x], reverse=True)

        top_ids = [all_evidence_ids[j] for j in ranked[:top_n]]

        if return_scores:
            top_scores = [score_map[j] for j in ranked[:top_n]]
            results.append((top_ids, top_scores))
        else:
            results.append(top_ids)
    return results


def build_stage1_candidates_for_processed_list(
    processed_retrieve_list,
    top_n=STAGE1_TOP_N_DEFAULT,
    verbose=False,
    show_progress_bar=True,
    claim_batch_size=STAGE1_CLAIM_BATCH_SIZE,
):
    """Build Stage1 top-N candidates using a processed retrieval list.

    Expected item format in the list: {claim_id, claim_text}.
    """
    topk_map = {}
    proc_list = list(processed_retrieve_list)
    n = len(proc_list)
    batch_starts = list(range(0, n, claim_batch_size))
    iterator = batch_starts
    if show_progress_bar:
        iterator = tqdm(batch_starts, desc="Stage1 retrieval", unit="batch")

    for start in iterator:
        chunk = proc_list[start : start + claim_batch_size]
        claim_ids = [item["claim_id"] for item in chunk]
        claim_texts = [item["claim_text"] for item in chunk]
        batch_out = retrieve_stage1_candidates(claim_texts, top_n=top_n, return_scores=False)
        for claim_id, top_ids in zip(claim_ids, batch_out):
            topk_map[claim_id] = top_ids
            if verbose:
                print(f"[Stage1] claim_id={claim_id} candidates={len(top_ids)}")
    return topk_map

In [ ]:
# -------------------------
# Stage1 run on train retrieval list
# -------------------------
load_stage1_candidates = False
stage1_output_path = data_path.parent / "train_stage1_topk.json"
# if we have processed data, load it
if load_stage1_candidates:
    with open(stage1_output_path, "r", encoding="utf-8") as f:
        train_stage1_topk = json.load(f)
    print("Loaded Stage1 candidates from:", stage1_output_path)
else:
    train_stage1_topk = build_stage1_candidates_for_processed_list(
        processed_train_retrieve_data,
        top_n=STAGE1_TOP_N_DEFAULT,
        verbose=False,
    )
    with open(stage1_output_path, "w", encoding="utf-8") as f:
        json.dump(train_stage1_topk, f, ensure_ascii=False, indent=2)
    print("Saved Stage1 candidates to:", stage1_output_path)

print("Stage1 candidates ready for train claims:", len(train_stage1_topk))

In [ ]:
# How many labeled gold evidences fall inside Stage1 top-N (train recall diagnostic).
gold_hits = 0
gold_total = 0
claims_with_gold = 0
claims_all_gold_in_stage1 = 0
macro_recall_sum = 0.0
for claim_id, item in train_claims.items():
    gold_ids = [eid for eid in item.get("evidences", []) if eid]
    if not gold_ids:
        continue
    claims_with_gold += 1
    cand_set = set(train_stage1_topk.get(claim_id, []))
    hits = sum(1 for eid in gold_ids if eid in cand_set)
    gold_hits += hits
    gold_total += len(gold_ids)
    macro_recall_sum += hits / len(gold_ids)
    if hits == len(gold_ids):
        claims_all_gold_in_stage1 += 1

if gold_total > 0:
    print(
        "Train: gold evids inside Stage1 candidates (micro):",
        f"{gold_hits}/{gold_total} = {gold_hits / gold_total:.4f}",
    )
if claims_with_gold > 0:
    print(
        "Train: mean per-claim fraction of gold inside Stage1 (macro):",
        f"{macro_recall_sum / claims_with_gold:.4f}",
    )
    print(
        "Train: claims with all gold inside Stage1:",
        f"{claims_all_gold_in_stage1}/{claims_with_gold} = {claims_all_gold_in_stage1 / claims_with_gold:.4f}",
    )

In [ ]:
# -------------------------
# Stage1 run on dev retrieval list
# -------------------------
load_stage1_candidates = False
stage1_output_path_dev = data_path.parent / "dev_stage1_topk.json"
# if we have processed data, load it
if load_stage1_candidates:
    with open(stage1_output_path_dev, "r", encoding="utf-8") as f:
        dev_stage1_topk = json.load(f)
    print("Loaded Stage1 candidates from:", stage1_output_path_dev)
else:
    dev_stage1_topk = build_stage1_candidates_for_processed_list(
        processed_dev_retrieve_data,
        top_n=STAGE1_TOP_N_DEFAULT,
        verbose=False,
    )
    with open(stage1_output_path_dev, "w", encoding="utf-8") as f:
        json.dump(dev_stage1_topk, f, ensure_ascii=False, indent=2)
    print("Saved Stage1 candidates to:", stage1_output_path_dev)

print("Stage1 candidates ready for train claims:", len(dev_stage1_topk))

In [ ]:
# How many labeled gold evidences fall inside Stage1 top-N (train recall diagnostic).
gold_hits = 0
gold_total = 0
claims_with_gold = 0
claims_all_gold_in_stage1 = 0
macro_recall_sum = 0.0
for claim_id, item in dev_claims.items():
    gold_ids = [eid for eid in item.get("evidences", []) if eid]
    if not gold_ids:
        continue
    claims_with_gold += 1
    cand_set = set(dev_stage1_topk.get(claim_id, []))
    hits = sum(1 for eid in gold_ids if eid in cand_set)
    gold_hits += hits
    gold_total += len(gold_ids)
    macro_recall_sum += hits / len(gold_ids)
    if hits == len(gold_ids):
        claims_all_gold_in_stage1 += 1

if gold_total > 0:
    print(
        "Train: gold evids inside Stage1 candidates (micro):",
        f"{gold_hits}/{gold_total} = {gold_hits / gold_total:.4f}",
    )
if claims_with_gold > 0:
    print(
        "Train: mean per-claim fraction of gold inside Stage1 (macro):",
        f"{macro_recall_sum / claims_with_gold:.4f}",
    )
    print(
        "Train: claims with all gold inside Stage1:",
        f"{claims_all_gold_in_stage1}/{claims_with_gold} = {claims_all_gold_in_stage1 / claims_with_gold:.4f}",
    )

### 2.1.2 Stage 2: Reranking top-k evidences

In [ ]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
batch_size=16
num_epochs=3  # improved: was 1
learning_rate=2e-5
warmup_ratio=0.1
output_path="./stage2_reranker_ckpt"


In [ ]:
def build_stage2_training_examples(
    claims_dict,
    stage1_topk_map,
    negatives_per_claim=4,
    seed=42,
):
    """Create pairwise training data for CrossEncoder reranker.

    Positive pairs: (claim, gold evidence) label=1
    Negative pairs: (claim, non-gold evidence from stage1 candidates) label=0
    """
    if InputExample is None:
        raise ImportError("sentence-transformers is required. Run: pip install sentence-transformers")

    rng = random.Random(seed)
    examples = []

    for claim_id, item in claims_dict.items():
        claim_text = clean_text(item["claim_text"])
        gold_ids = set(item.get("evidences", []))
        candidates = stage1_topk_map.get(claim_id, [])

        # positives
        for eid in gold_ids:
            ev_text = evidence_id_to_text.get(eid)
            if ev_text is not None:
                examples.append(InputExample(texts=[claim_text, ev_text], label=1.0))

        # negatives: random sample from stage1 candidates that are not gold
        hard_neg_ids = [eid for eid in candidates if eid not in gold_ids]
        if len(hard_neg_ids) > negatives_per_claim:
            hard_neg_ids = rng.sample(hard_neg_ids, negatives_per_claim)

        for eid in hard_neg_ids:
            ev_text = evidence_id_to_text.get(eid)
            if ev_text is not None:
                examples.append(InputExample(texts=[claim_text, ev_text], label=0.0))

    return examples

stage2_train_examples = build_stage2_training_examples(
    train_claims,
    train_stage1_topk,
    negatives_per_claim=40,
    seed=42,
)

In [ ]:
reranker = CrossEncoder(RERANKER_MODEL_NAME, num_labels=1)
train_loader = DataLoader(stage2_train_examples, shuffle=True, batch_size=batch_size)
warmup_steps = max(1, int(len(train_loader) * num_epochs * warmup_ratio))
reranker.fit(
    train_dataloader=train_loader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": learning_rate},
    output_path=output_path,
    show_progress_bar=True,
)
reranker



In [ ]:
# =============================================================================
# NEW: Build claim-level records with RETRIEVED evidence (+ reranker scores)
# =============================================================================
# Hybrid notebook trained BERT on one string with ALL passages concatenated.
# This notebook needs per-passage inputs, so we first run the full retriever on
# each train claim and store (evidence_ids, evidence_scores) per claim.

train_retrieved_records = [
    {"claim_id": cid, "claim_text": clean_text(train_claims[cid]["claim_text"])}
    for cid in train_claims
]

print("Building retrieved training claims for classifier...")
_ = build_retrieval_predictions(
    train_retrieved_records,
    top_k=RERANK_TOP_K_DEFAULT,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    reranker=reranker,
    rerank_batch_size=32,
    show_progress_bar=True,
    verbose=False,
)

# Structured train rows: label is still GOLD from train-claims.json
processed_train_classifier = []
for row in train_retrieved_records:
    processed_train_classifier.append({
        "claim_id": row["claim_id"],
        "claim_text": row["claim_text"],
        "evidences": list(row["evidences"]),
        "evidence_scores": list(row.get("evidence_scores", [1.0] * len(row["evidences"]))),
        "label": train_claims[row["claim_id"]]["claim_label"],
    })
print("Retrieved train claims:", len(processed_train_classifier))

# Dev retrieved rows (built in previous cell as dev_retrieval_records) + gold labels for eval
processed_dev_retrieved = []
for row in dev_retrieval_records:
    processed_dev_retrieved.append({
        "claim_id": row["claim_id"],
        "claim_text": row["claim_text"],
        "evidences": list(row["evidences"]),
        "evidence_scores": list(row.get("evidence_scores", [1.0] * len(row["evidences"]))),
        "label": dev_claims[row["claim_id"]]["claim_label"],
    })
print("Retrieved dev claims:", len(processed_dev_retrieved))


In [ ]:
def build_retrieval_predictions(
    processed_retrieve_list,
    top_k=RERANK_TOP_K_DEFAULT,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    reranker=None,
    rerank_batch_size=32,
    verbose=True,
    show_progress_bar=True,
):
    """Stage 1 + rerank.

    NEW vs hybrid notebook: also stores `evidence_scores` (cross-encoder scores).
    Those scores are used later to weight each passage when aggregating BERT logits.
    """
    proc_list = list(processed_retrieve_list)
    n = len(proc_list)

    # Phase 1: hybrid TF-IDF + dense top-N (unchanged)
    stage1_topk_map = build_stage1_candidates_for_processed_list(
        proc_list,
        top_n=stage1_top_n,
        verbose=False,
        show_progress_bar=show_progress_bar,
    )

    phase2_iter = range(n)
    if show_progress_bar:
        phase2_iter = tqdm(range(n), desc="Stage2 rerank", unit="claim")

    for idx in phase2_iter:
        item = proc_list[idx]
        claim_id = item["claim_id"]
        claim_text = item["claim_text"]
        stage1_ids = stage1_topk_map.get(claim_id, [])

        if reranker is None:
            top_ids = stage1_ids[:top_k]
            item["evidences"] = top_ids
            item["evidence_scores"] = [1.0] * len(top_ids)  # no reranker => equal weights
        else:
            claim_clean = clean_text(claim_text)
            cand_ids = [eid for eid in stage1_ids if eid in evidence_id_to_text]
            pairs = [[claim_clean, evidence_id_to_text[eid]] for eid in cand_ids]
            if not pairs:
                top_ids = stage1_ids[:top_k]
                item["evidences"] = top_ids
                item["evidence_scores"] = [1.0] * len(top_ids)
            else:
                scores = reranker.predict(
                    pairs, batch_size=rerank_batch_size, show_progress_bar=False
                )
                ranked = sorted(
                    zip(cand_ids, scores), key=lambda x: float(x[1]), reverse=True
                )
                top = ranked[:top_k]
                item["evidences"] = [eid for eid, _ in top]
                # NEW: keep reranker score per kept passage for weighted aggregation
                item["evidence_scores"] = [float(s) for _, s in top]

        if verbose:
            mode = "two-stage" if reranker is not None else "stage1-only"
            print(f"[{mode}] claim_id={claim_id} n_ev={len(item.get('evidences', []))}")

    return proc_list


stage2_reranker = reranker  # alias used by pipeline / test cells below

# Run retrieval on DEV once here (used for dev eval + processed_dev_retrieved)
dev_retrieval_records = build_retrieval_predictions(
    copy.deepcopy(processed_dev_retrieve_data),
    top_k=RERANK_TOP_K_DEFAULT,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    reranker=reranker,
    rerank_batch_size=32,
    verbose=False,
)
print("Dev retrieval done:", len(dev_retrieval_records))
if dev_retrieval_records:
    print("Sample:", {k: dev_retrieval_records[0].get(k) for k in ("claim_id", "evidences", "evidence_scores")})


In [ ]:
def official_evidence_retrieval_f_only(predictions, groundtruth, verbose=False):
    """Evidence retrieval F-score only (same definition as eval.py for F).

    predictions: {claim_id: {"evidences": [...]}} (other keys ignored).
    groundtruth: {claim_id: {..., "evidences": [...]}}
    """
    f_scores = []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "evidences" not in pred:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0
        if isinstance(pred["evidences"], list) and len(pred["evidences"]) > 0:
            pred_set = set(pred["evidences"])
            for gr_ev in claim["evidences"]:
                if gr_ev in pred_set:
                    evidence_correct += 1
            if evidence_correct > 0:
                evidence_recall = float(evidence_correct) / len(claim["evidences"])
                evidence_precision = float(evidence_correct) / len(pred["evidences"])
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                    evidence_precision + evidence_recall
                )

        if verbose:
            print("claim_id =", claim_id)
            print("groundtruth evidences =", claim["evidences"])
            print("predicted evidences   =", pred.get("evidences"))
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n")

        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    return mean_f


# Map dev retrieval list -> eval-style dict (retrieval only).
dev_retrieval_pred_dict = {
    row["claim_id"]: {"evidences": list(row["evidences"])}
    for row in dev_retrieval_records
}

F_retrieval_dev = official_evidence_retrieval_f_only(
    dev_retrieval_pred_dict, dev_claims, verbose=False
)
print("Dev Evidence Retrieval F-score =", F_retrieval_dev)

## 2.2 Classification

In [ ]:
# =============================================================================
# NEW: Per-evidence classification helpers
# =============================================================================
# OLD (hybrid): one BERT input = claim + bullet list of ALL evidences.
# NEW: one BERT input = claim + ONE evidence passage (a "pair" row).

def format_pair_input(claim_text, evidence_text):
    """Single-passage input string fed to BERT."""
    claim_text = clean_text(claim_text)
    evidence_text = clean_text(evidence_text)
    return f"Claim: {claim_text}\n\nEvidence:\n- {evidence_text}"


def expand_claims_to_pair_rows(claim_records):
    """Expand each claim into N rows (one per retrieved/gold passage).

    Example: claim with 5 evidences -> 5 training rows, same gold label each time.
    """
    rows = []
    for rec in claim_records:
        if not rec.get("label"):
            continue
        claim_text = rec["claim_text"]
        eids = rec.get("evidences", [])
        scores = rec.get("evidence_scores", [1.0] * len(eids))
        if len(scores) != len(eids):
            scores = [1.0] * len(eids)
        for eid, sc in zip(eids, scores):
            if eid not in evidence:
                continue
            rows.append({
                "text": format_pair_input(claim_text, evidence[eid]),
                "label": rec["label"],  # gold label from JSON (supervision)
                "claim_id": rec["claim_id"],
            })
    return rows


def build_mixed_pair_rows(gold_claim_records, retrieved_claim_records, gold_ratio=MIXED_TRAIN_RATIO, seed=42):
    """Mix gold-evidence pairs + retrieved-evidence pairs for training.

    Why mix?
    - Retrieved-only: matches test, but can be very noisy.
    - Gold-only: easy passages, poor match to inference (hybrid problem).
    - Mix: balance realism and learnable signal.
    """
    gold_pairs = expand_claims_to_pair_rows(gold_claim_records)
    retrieved_pairs = expand_claims_to_pair_rows(retrieved_claim_records)
    rng = random.Random(seed)
    # Subsample gold pairs so gold_ratio of total ~= MIXED_TRAIN_RATIO
    n_gold = int(len(retrieved_pairs) * gold_ratio / max(1e-9, (1.0 - gold_ratio)))
    n_gold = min(n_gold, len(gold_pairs))
    if n_gold < len(gold_pairs):
        gold_pairs = rng.sample(gold_pairs, n_gold)
    mixed = gold_pairs + retrieved_pairs
    rng.shuffle(mixed)
    print(f"Mixed pair training: {len(gold_pairs)} gold + {len(retrieved_pairs)} retrieved = {len(mixed)} rows")
    return mixed


def pairs_to_hf_dataset(pair_rows):
    return Dataset.from_dict({
        "text": [r["text"] for r in pair_rows],
        "labels": [label2id[r["label"]] for r in pair_rows],
    })


# BERT classifier (same backbone as hybrid — design change is HOW we call it)
MODEL_NAME = "google-bert/bert-base-uncased"
LABEL_LIST = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
label2id = {name: i for i, name in enumerate(LABEL_LIST)}
id2label = {i: name for name, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)


In [ ]:
# =============================================================================
# NEW: Build HuggingFace datasets from PAIR rows (not one row per claim)
# =============================================================================

train_pair_rows = build_mixed_pair_rows(
    processed_train_data,           # gold passages from train-claims.json
    processed_train_classifier,     # retrieved passages from our pipeline
    gold_ratio=MIXED_TRAIN_RATIO,
    seed=42,
)
train_hf = pairs_to_hf_dataset(train_pair_rows)

# Validation during training: retrieved dev pairs if flag on (recommended)
if USE_RETRIEVED_DEV_EVAL:
    dev_pair_rows = expand_claims_to_pair_rows(processed_dev_retrieved)
    print("Dev eval: retrieved pair rows =", len(dev_pair_rows))
else:
    dev_pair_rows = expand_claims_to_pair_rows(processed_dev_data)
    print("Dev eval: gold pair rows =", len(dev_pair_rows))

dev_hf = pairs_to_hf_dataset(dev_pair_rows)


def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

train_tok = train_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
dev_tok = dev_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    # Note: this is PAIR-level accuracy/F1 (one row per passage), not claim-level.
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        **accuracy_metric.compute(predictions=preds, references=labels),
        **f1_metric.compute(predictions=preds, references=labels, average="macro"),
    }


# NEW: class weights — penalise mistakes on rare labels more heavily
class_weight_tensor = None
if USE_CLASS_WEIGHTS:
    y = [label2id[r["label"]] for r in train_pair_rows]
    classes = np.arange(len(LABEL_LIST))
    from sklearn.utils.class_weight import compute_class_weight
    cw = compute_class_weight("balanced", classes=classes, y=y)
    class_weight_tensor = torch.tensor(cw, dtype=torch.float32)
    print("Class weights:", {LABEL_LIST[i]: round(float(cw[i]), 3) for i in range(len(LABEL_LIST))})


class WeightedTrainer(Trainer):
    """Trainer with optional weighted CrossEntropyLoss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if class_weight_tensor is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weight_tensor.to(logits.device))
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, len(LABEL_LIST)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


CLASSIFIER_OUTPUT_DIR = "./bert_per_evidence_ckpt"

training_args = TrainingArguments(
    output_dir=str(CLASSIFIER_OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    metric_for_best_model="f1",  # macro-F1 on pair-level dev rows
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print(f"train on {model.device}")
trainer.train()


In [ ]:
# =============================================================================
# NEW: Inference — score each passage, then aggregate (claim-level label)
# =============================================================================

def predict_label_aggregate(claim_record, batch_size=16):
    """Run BERT on each (claim, passage) pair, merge logits using reranker weights.

    Steps:
      1) Build one input string per evidence passage
      2) BERT forward -> shape (num_passages, 4) logits
      3) Convert reranker scores to weights via softmax
      4) Weighted sum of logits -> single 4-d vector -> argmax label

    This replaces hybrid's single forward pass on concatenated evidence.
    """
    claim_text = claim_record["claim_text"]
    eids = list(claim_record.get("evidences", []))
    scores = list(claim_record.get("evidence_scores", [1.0] * len(eids)))
    if not eids:
        return "NOT_ENOUGH_INFO"

    texts, valid_scores = [], []
    for eid, sc in zip(eids, scores):
        if eid in evidence_id_to_text:
            texts.append(format_pair_input(claim_text, evidence_id_to_text[eid]))
            valid_scores.append(float(sc))

    if not texts:
        return "NOT_ENOUGH_INFO"

    model.eval()
    all_logits = []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start : start + batch_size]
        inputs = tokenizer(
            chunk, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits.cpu().numpy()
        all_logits.append(logits)
    logits = np.concatenate(all_logits, axis=0)  # (num_passages, 4)

    # Softmax over reranker scores: higher-confidence passages vote more
    w = np.array(valid_scores, dtype=np.float64)
    w = w - w.max()  # numerical stability
    w = np.exp(w)
    w = w / (w.sum() + 1e-12)
    agg = (logits * w[:, None]).sum(axis=0)
    return id2label[int(agg.argmax())]


def build_pipeline_predictions(retrieval_records, batch_size=16):
    """One claim-level prediction dict per claim (for eval.py format)."""
    records = list(retrieval_records)
    out = {}
    for rec in sorted(records, key=lambda x: x["claim_id"]):
        lab = predict_label_aggregate(rec, batch_size=batch_size)
        out[rec["claim_id"]] = {
            "claim_text": rec["claim_text"],
            "claim_label": lab,
            "evidences": list(rec.get("evidences", [])),
        }
    return out


def run_retrieval_then_classification(
    processed_retrieve_records,
    top_k=RERANK_TOP_K_DEFAULT,
    batch_size=16,
    verbose=False,
    reranker=None,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
    show_progress_bar=True,
):
    print("Retrieval start:")
    retrieval_records = build_retrieval_predictions(
        processed_retrieve_records,
        top_k=top_k,
        stage1_top_n=stage1_top_n,
        reranker=reranker,
        rerank_batch_size=rerank_batch_size,
        verbose=verbose,
        show_progress_bar=show_progress_bar,
    )
    print("Retrieval End:")
    print("Classification (per-evidence + aggregation):")
    return build_pipeline_predictions(retrieval_records, batch_size=batch_size)


# Diagnostic: ceiling test — use GOLD dev passages but SAME aggregation logic
_dev_gold_agg = build_pipeline_predictions(
    [
        {
            **r,
            "evidences": dev_claims[r["claim_id"]]["evidences"],
            "evidence_scores": [1.0] * len(dev_claims[r["claim_id"]]["evidences"]),
        }
        for r in processed_dev_retrieve_data
    ],
    batch_size=16,
)
print(
    "Dev accuracy (gold evidence, aggregated):",
    sum(
        _dev_gold_agg[r["claim_id"]]["claim_label"] == dev_claims[r["claim_id"]]["claim_label"]
        for r in processed_dev_data
    ) / len(processed_dev_data),
)


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# Experiment log (dev) — copy the table into your report
import pandas as pd

EXPERIMENT_LOG = globals().get("EXPERIMENT_LOG", [])

def log_experiment(name, F=None, A=None, H=None, notes=""):
    global EXPERIMENT_LOG
    if F is None or A is None or H is None:
        F, A, H = official_eval_like_script(dev_pipeline_predictions, dev_claims)
    EXPERIMENT_LOG.append({"exp": name, "F": round(F, 4), "A": round(A, 4), "H": round(H, 4), "notes": notes})
    return pd.DataFrame(EXPERIMENT_LOG)


In [ ]:
# CLAIM-LEVEL dev evaluation (F, A, H) — uses retrieval + aggregation above
import numpy as np


def official_eval_like_script(predictions, groundtruth, verbose=False):
    """Same metrics as eval.py. Returns (F, A, harmonic_mean)."""
    f_scores, acc = [], []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue

        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0
        if isinstance(pred["evidences"], list) and len(pred["evidences"]) > 0:
            pred_set = set(pred["evidences"])
            for gr_ev in claim["evidences"]:
                if gr_ev in pred_set:
                    evidence_correct += 1
            if evidence_correct > 0:
                evidence_recall = float(evidence_correct) / len(claim["evidences"])
                evidence_precision = float(evidence_correct) / len(pred["evidences"])
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                    evidence_precision + evidence_recall
                )

        if verbose:
            print("groundtruth =", claim)
            print("predictions =", pred)
            print("instance accuracy =", instance_correct)
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n\n")

        acc.append(instance_correct)
        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    mean_acc = float(np.mean(acc if len(acc) > 0 else [0.0]))
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return mean_f, mean_acc, hmean


dev_pipeline_predictions = run_retrieval_then_classification(
    processed_dev_retrieve_data,
    top_k=RERANK_TOP_K_DEFAULT,
    batch_size=16,
    verbose=False,
    reranker=stage2_reranker,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
)

F, A, H = official_eval_like_script(
    dev_pipeline_predictions, dev_claims, verbose=False
)
print("Evidence Retrieval F-score (F)    =", F)
print("Claim Classification Accuracy (A) =", A)
print("Harmonic Mean of F and A          =", H)

display(log_experiment("per_evidence_agg_dev", F=F, A=A, H=H, notes="per-evidence BERT + score aggregation, mixed train, class weights"))


In [ ]:
# Per-label error analysis (dev pipeline)
from collections import Counter
from sklearn.metrics import classification_report

y_true, y_pred = [], []
for row in processed_dev_data:
    cid = row["claim_id"]
    y_true.append(dev_claims[cid]["claim_label"])
    y_pred.append(dev_pipeline_predictions[cid]["claim_label"])

print("Classification report (claim-level, dev pipeline):")
print(classification_report(y_true, y_pred, labels=LABEL_LIST, zero_division=0))

wrong = [(t, p) for t, p in zip(y_true, y_pred) if t != p]
print("Top confusion pairs (gold -> pred):", Counter(wrong).most_common(10))
print("Pipeline accuracy:", sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true))


In [ ]:
# Optional: retrain on train+dev pair rows (final test only)
if USE_TRAIN_PLUS_DEV_FOR_FINAL:
    final_claims = {**train_claims, **dev_claims}
    final_retrieve = [
        {"claim_id": cid, "claim_text": clean_text(final_claims[cid]["claim_text"])}
        for cid in final_claims
    ]
    print("Building final train+dev retrieved claims...")
    _ = build_retrieval_predictions(
        final_retrieve,
        top_k=RERANK_TOP_K_DEFAULT,
        stage1_top_n=STAGE1_TOP_N_DEFAULT,
        reranker=reranker,
        rerank_batch_size=32,
        show_progress_bar=True,
        verbose=False,
    )
    final_gold = process_data(final_claims, classification_train=True)
    final_retrieved = []
    for row in final_retrieve:
        final_retrieved.append({
            "claim_id": row["claim_id"],
            "claim_text": row["claim_text"],
            "evidences": list(row["evidences"]),
            "evidence_scores": list(row.get("evidence_scores", [1.0] * len(row["evidences"]))),
            "label": final_claims[row["claim_id"]]["claim_label"],
        })
    final_pair_rows = build_mixed_pair_rows(final_gold, final_retrieved, gold_ratio=MIXED_TRAIN_RATIO)
    final_hf = pairs_to_hf_dataset(final_pair_rows)
    final_tok = final_hf.map(tokenize_batch, batched=True, remove_columns=["text"])

    final_args = TrainingArguments(
        output_dir="./bert_per_evidence_final_ckpt",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        logging_steps=50,
        seed=42,
        report_to="none",
    )
    final_trainer = WeightedTrainer(
        model=model,
        args=final_args,
        train_dataset=final_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
    )
    final_trainer.train()
    print("Final per-evidence classifier trained on train+dev mixed pairs.")
else:
    print("USE_TRAIN_PLUS_DEV_FOR_FINAL is False — using dev-tuned checkpoint for test.")


In [ ]:
import json

# Test (unlabeled): run retrieval + classification and save as JSON.
test_predictions = run_retrieval_then_classification(
    processed_test_data,
    top_k=RERANK_TOP_K_DEFAULT,
    batch_size=16,
    verbose=False,
    reranker=stage2_reranker,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
)

output_path = data_path.parent / "test-output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

print("Saved test predictions to:", output_path)
print("Number of test claims:", len(test_predictions))

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*